# 2.3. Подготовка данных

**Цель этапа:** выполнить предварительный анализ, очистку, аннотирование/векторизацию неструктурированных данных, определить шкалы, аналитические роли и общие атрибуты, обосновать структуру хранения и выполнить слияние разнородных данных (п.4 и п.5 методических указаний).

**План:**
1. Очистка Retail (транзакции).
2. Очистка XML (регионы РФ).
3. Очистка Walmart (JSON).
4. Очистка и аннотирование Reviews (текст).
5. Таблица шкал и аналитических ролей.
6. Обоснование структуры хранения.
7. Слияние по оси «Время».
8. Замеры времени на каждом шаге.

In [14]:
import pandas as pd
import numpy as np
import time
import re
import json
import sqlite3
import xml.etree.ElementTree as ET
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Универсальный таймер
def timed(label, func, *args, **kwargs):
    t0 = time.time()
    result = func(*args, **kwargs)
    dt = time.time() - t0
    print(f"[TIME] {label}: {dt:.2f} сек")
    return result, dt

timings = {}

In [15]:
# Retail
t0 = time.time()
df_retail = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011', engine='openpyxl')
timings['load_retail'] = time.time() - t0

# XML (парсинг уже был в 2.2 — переиспользуем функцию)
def parse_xml(path):
    tree = ET.parse(path); root = tree.getroot()
    ns = {'generic': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/generic',
          'structure': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/structure'}
    okato = {}
    cl = root.find('.//structure:CodeList[@id="s_OKATO"]', ns)
    if cl is not None:
        for c in cl.findall('structure:Code', ns):
            d = c.find('structure:Description', ns)
            if d is not None: okato[c.get('value')] = d.text
    recs = []
    for s in root.findall('.//generic:Series', ns):
        sk = s.find('generic:SeriesKey', ns)
        ok = sk.find('generic:Value', ns).get('value') if sk is not None else None
        period = unit = None
        at = s.find('generic:Attributes', ns)
        if at is not None:
            for v in at.findall('generic:Value', ns):
                if v.get('concept') == 'PERIOD': period = v.get('value')
                if v.get('concept') == 'EI': unit = v.get('value')
        for o in s.findall('generic:Obs', ns):
            tm = o.find('generic:Time', ns); vl = o.find('generic:ObsValue', ns)
            recs.append({'okato': ok, 'region': okato.get(ok, 'Неизвестно'),
                         'period': period, 'year': tm.text if tm is not None else None,
                         'value': float(vl.get('value').replace(',', '.')) if vl is not None else None,
                         'unit': unit})
    return pd.DataFrame(recs)

t0 = time.time()
df_xml = parse_xml('data.xml')
timings['load_xml'] = time.time() - t0

# Walmart (JSON)
t0 = time.time()
with open('walmart.json', 'r', encoding='utf-8') as f:
    df_api = pd.json_normalize(json.load(f))
df_api['Date'] = pd.to_datetime(df_api['Date'], format='%d-%m-%Y')
df_api['Month'] = df_api['Date'].dt.month
df_api['Store'] = df_api['Store'].astype(str)
timings['load_walmart'] = time.time() - t0

# Reviews
t0 = time.time()
df_reviews = pd.read_csv('Reviews.csv')
df_reviews['review_date'] = pd.to_datetime(df_reviews['Time'], unit='s')
timings['load_reviews'] = time.time() - t0

print("Загружено:", df_retail.shape, df_xml.shape, df_api.shape, df_reviews.shape)

Загружено: (541910, 8) (368, 6) (6435, 9) (568454, 11)


## 1. Очистка Retail

**Что делаем и зачем:**
- Удаляем отмены (`Invoice` начинается с `C`) — это возвраты, они искажают выручку.
- Убираем `Quantity <= 0` и `Price <= 0` — служебные корректировки/ошибки.
- Убираем дубликаты строк.
- Оставляем только строки с известным `Customer ID` — нужны для сегментации клиентов (RFM в 2.4).
- Добавляем `Revenue = Quantity * Price`.
- Извлекаем `Year`, `Month`, `DayOfWeek`, `Hour` — для временного анализа.

In [16]:
def clean_retail(df):
    n0 = len(df)
    df = df.copy()
    df['Invoice'] = df['Invoice'].astype(str)
    # 1. Отмены
    df = df[~df['Invoice'].str.startswith('C')]
    # 2. Отрицательные/нулевые
    df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
    # 3. Дубликаты
    df = df.drop_duplicates()
    # 4. Только с Customer ID
    df = df[df['Customer ID'].notna()].copy()
    df['Customer ID'] = df['Customer ID'].astype(int)
    # 5. Выручка
    df['Revenue'] = df['Quantity'] * df['Price']
    # 6. Временные признаки
    df['Year'] = df['InvoiceDate'].dt.year
    df['Month'] = df['InvoiceDate'].dt.month
    df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
    df['Hour'] = df['InvoiceDate'].dt.hour
    df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
    print(f"Retail: {n0} -> {len(df)} строк (удалено {n0-len(df)})")
    return df

df_retail_clean, timings['clean_retail'] = timed('clean_retail', clean_retail, df_retail)

print(df_retail_clean[['Invoice','Quantity','Price','Revenue','YearMonth']].head())
print("\nПроверка:")
print("  Min Revenue:", df_retail_clean['Revenue'].min())
print("  Уник. клиентов:", df_retail_clean['Customer ID'].nunique())
print("  Период:", df_retail_clean['InvoiceDate'].min(), "—", df_retail_clean['InvoiceDate'].max())

Retail: 541910 -> 392693 строк (удалено 149217)
[TIME] clean_retail: 1.59 сек
  Invoice  Quantity  Price  Revenue YearMonth
0  536365         6   2.55    15.30   2010-12
1  536365         6   3.39    20.34   2010-12
2  536365         8   2.75    22.00   2010-12
3  536365         6   3.39    20.34   2010-12
4  536365         6   3.39    20.34   2010-12

Проверка:
  Min Revenue: 0.001
  Уник. клиентов: 4338
  Период: 2010-12-01 08:26:00 — 2011-12-09 12:50:00


## 2. Очистка XML (ЕМИСС/Росстат)

**Что делаем и зачем:**
- `year` → int, `value` → float (уже float, но проверяем).
- Убираем агрегаты: «Российская Федерация», федеральные округа (коды 030, 031…, 643). Оставляем только регионы (ОКАТО из 11 цифр или 2-значные коды регионов).
- Нормализуем `period` в `Jan-Jun` / `Jan-Dec`.
- `unit` = «тысяча рублей» → переводим в рубли (×1000) для сопоставимости.

In [17]:
def clean_xml(df):
    n0 = len(df)
    df = df.copy()
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df.dropna(subset=['year', 'value'])
    # Оставляем только регионы (исключаем РФ и федеральные округа)
    mask_fed = df['region'].str.contains('федеральный округ', case=False, na=False)
    mask_rf = df['region'].str.contains('Российская Федерация', case=False, na=False)
    df = df[~mask_fed & ~mask_rf].copy()
    # Нормализация периода
    df['period'] = df['period'].map({'январь-июнь': 'H1', 'январь-декабрь': 'H2'})
    # Перевод в рубли
    df['value_rub'] = df['value'] * 1000
    print(f"XML: {n0} -> {len(df)} строк (удалено {n0-len(df)})")
    return df

df_xml_clean, timings['clean_xml'] = timed('clean_xml', clean_xml, df_xml)

print(df_xml_clean[['region','period','year','value_rub']].head())
print("\nУник. регионов:", df_xml_clean['region'].nunique())

XML: 368 -> 332 строк (удалено 36)
[TIME] clean_xml: 0.01 сек
                  region period  year     value_rub
8   Белгородская область     H1  2010  5.020594e+09
9   Белгородская область     H2  2010  1.054261e+10
10  Белгородская область     H1  2011  4.286433e+09
11  Белгородская область     H2  2011  7.536352e+09
12      Брянская область     H1  2010  2.757214e+09

Уник. регионов: 83


## 3. Очистка Walmart (JSON)

**Что делаем и зачем:**
- Даты уже распарсены, `Month` получен.
- Убираем строки с `Weekly_Sales <= 0`.
- Проверяем выбросы по IQR для `Weekly_Sales` (не удаляем, а помечаем флагом `is_outlier` — пригодится в 2.4).
- Добавляем `Year` и `YearMonth` для слияния по времени.

In [18]:
def clean_walmart(df):
    n0 = len(df)
    df = df.copy()
    df = df[df['Weekly_Sales'] > 0]
    # Флаг выброса по IQR
    q1, q3 = df['Weekly_Sales'].quantile([0.25, 0.75])
    iqr = q3 - q1
    df['is_outlier'] = (df['Weekly_Sales'] < q1 - 1.5*iqr) | (df['Weekly_Sales'] > q3 + 1.5*iqr)
    df['Year'] = df['Date'].dt.year
    df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
    print(f"Walmart: {n0} -> {len(df)} строк, выбросов: {df['is_outlier'].sum()}")
    return df

df_walmart_clean, timings['clean_walmart'] = timed('clean_walmart', clean_walmart, df_api)

print(df_walmart_clean[['Store','Date','Weekly_Sales','Year','Month','YearMonth']].head())

Walmart: 6435 -> 6435 строк, выбросов: 34
[TIME] clean_walmart: 0.01 сек
  Store       Date  Weekly_Sales  Year  Month YearMonth
0     1 2010-02-05    1643690.90  2010      2   2010-02
1     1 2010-02-12    1641957.44  2010      2   2010-02
2     1 2010-02-19    1611968.17  2010      2   2010-02
3     1 2010-02-26    1409727.59  2010      2   2010-02
4     1 2010-03-05    1554806.68  2010      3   2010-03


## 4. Очистка и аннотирование Reviews

**Что делаем и зачем:**
- Убираем пропуски `ProfileName`/`Summary` (единицы).
- `text_length`, `summary_length` уже посчитаны в 2.2.
- **Аннотирование тональности:** по `Score` → `sentiment` (neg: 1–2, neu: 3, pos: 4–5).
- **Векторизация текста:** TF-IDF по `Text` (ограничим 5000 признаками для скорости) + TruncatedSVD до 20 компонент — это числовые признаки для 2.4.

In [19]:
def clean_and_vectorize_reviews(df, n_samples=50000):
    df = df.copy()
    df = df.dropna(subset=['ProfileName', 'Summary'])

    # Длины текста
    df['text_length'] = df['Text'].astype(str).str.len()
    df['summary_length'] = df['Summary'].astype(str).str.len()

    # Тональность
    def to_sent(s):
        if s <= 2: return 'neg'
        if s == 3: return 'neu'
        return 'pos'
    df['sentiment'] = df['Score'].apply(to_sent)

    # Полный набор с тональностью (без SVD) — для временного анализа в 2.4
    df_full = df.copy()

    # Для скорости векторизуем подвыборку
    df_sample = df.sample(n=min(n_samples, len(df)), random_state=42).copy()

    t0 = time.time()
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
    X_tfidf = tfidf.fit_transform(df_sample['Text'].astype(str))
    timings['tfidf_reviews'] = time.time() - t0

    t0 = time.time()
    svd = TruncatedSVD(n_components=20, random_state=42)
    X_svd = svd.fit_transform(X_tfidf)
    timings['svd_reviews'] = time.time() - t0

    for i in range(20):
        df_sample[f'txt_svd_{i}'] = X_svd[:, i]

    print(f"Reviews: полный {len(df_full)}, семпл с SVD {len(df_sample)}")
    print(f"  TF-IDF: {X_tfidf.shape}, SVD: {X_svd.shape}")
    return df_full, df_sample, tfidf, svd

df_reviews_full, df_reviews_clean, tfidf_model, svd_model = clean_and_vectorize_reviews(df_reviews)

print("\nПолный набор (с тональностью):", df_reviews_full.shape)
print(df_reviews_full[['Score','sentiment','text_length']].head())
print("\nРаспределение тональности (полный):")
print(df_reviews_full['sentiment'].value_counts())

print("\nСэмпл с SVD:", df_reviews_clean.shape)
print("Примеры SVD-признаков:")
print(df_reviews_clean[['Score','sentiment','txt_svd_0','txt_svd_1']].head())

Reviews: полный 568401, семпл с SVD 50000
  TF-IDF: (50000, 5000), SVD: (50000, 20)

Полный набор (с тональностью): (568401, 14)
   Score sentiment  text_length
0      5       pos          263
1      1       neg          190
2      4       pos          509
3      2       neg          219
4      5       pos          140

Распределение тональности (полный):
sentiment
pos    443756
neg     82007
neu     42638
Name: count, dtype: int64

Сэмпл с SVD: (50000, 34)
Примеры SVD-признаков:
        Score sentiment  txt_svd_0  txt_svd_1
18828       5       pos   0.253701   0.129418
363857      4       pos   0.202294   0.106047
342609      1       neg   0.152234   0.091706
62213       5       pos   0.260499   0.014855
467133      5       pos   0.081568  -0.031768


## 5. Таблица шкал и аналитических ролей

Используем типологию из лекции «Аналитически ориентированная типология шкал атрибутов» (8 шкал): номинальная (ID), категориальная, порядковая, интервальная, отношений, абсолютная, дихотомическая, контентная.

**Аналитические роли в OLAP-концепции:**
- **Измерение (Dimension)** — ось агрегации.
- **Мера (Measure)** — числовой показатель для агрегации.
- **Ключ слияния (Join Key)** — общий атрибут.
- **Время** — ось для слияния.

In [20]:
scales_roles = pd.DataFrame([
    # Retail
    ('Retail','Invoice','Номинальная (ID)','Идентификатор заказа','Ключ (внутри Retail)'),
    ('Retail','StockCode','Номинальная (ID)','Код товара','Измерение'),
    ('Retail','Description','Контентная','Описание товара','—'),
    ('Retail','Quantity','Абсолютная','Количество единиц','Мера'),
    ('Retail','Price','Отношений','Цена за единицу','Мера'),
    ('Retail','Revenue','Отношений','Выручка = Qty*Price','Мера'),
    ('Retail','InvoiceDate','Интервальная','Дата/время покупки','Время'),
    ('Retail','Customer ID','Номинальная (ID)','ID клиента','Измерение'),
    ('Retail','Country','Категориальная','Страна','Измерение'),
    ('Retail','Year','Интервальная','Год','Ключ слияния'),
    ('Retail','Month','Категориальная','Месяц','Ключ слияния'),
    # XML
    ('XML','okato','Номинальная (ID)','Код ОКАТО','Измерение'),
    ('XML','region','Категориальная','Регион','Измерение'),
    ('XML','period','Категориальная','H1/H2','Измерение'),
    ('XML','year','Интервальная','Год','Ключ слияния'),
    ('XML','value_rub','Отношений','Оборот, руб.','Мера'),
    ('XML','xml_regions','Абсолютная','Число регионов','Мера'),
    # Walmart
    ('Walmart','Store','Категориальная','Номер магазина','Измерение'),
    ('Walmart','Date','Интервальная','Дата недели','Время'),
    ('Walmart','Weekly_Sales','Отношений','Недельные продажи','Мера'),
    ('Walmart','Holiday_Flag','Дихотомическая','Праздничная неделя','Измерение'),
    ('Walmart','is_outlier','Дихотомическая','Флаг выброса','Измерение'),
    ('Walmart','Month','Категориальная','Месяц','Ключ слияния'),
    ('Walmart','Year','Интервальная','Год','Ключ слияния'),
    ('Walmart','Temperature','Отношений','Температура','Фактор'),
    ('Walmart','Fuel_Price','Отношений','Цена топлива','Фактор'),
    ('Walmart','CPI','Отношений','ИПЦ','Фактор'),
    ('Walmart','Unemployment','Отношений','Безработица','Фактор'),
    # Reviews
    ('Reviews','ProductId','Номинальная (ID)','ID товара','Измерение'),
    ('Reviews','UserId','Номинальная (ID)','ID пользователя','Измерение'),
    ('Reviews','Score','Порядковая','Оценка 1–5','Измерение (порядковое)'),
    ('Reviews','sentiment','Категориальная','Тональность','Измерение'),
    ('Reviews','review_date','Интервальная','Дата отзыва','Время'),
    ('Reviews','text_length','Абсолютная','Длина текста','Мера'),
    ('Reviews','summary_length','Абсолютная','Длина резюме','Мера'),
    ('Reviews','txt_svd_0','Отношений','SVD-компонента','Мера'),
], columns=['Источник','Атрибут','Шкала','Смысл','Аналитическая роль'])

print(scales_roles.to_string(index=False))

Источник        Атрибут            Шкала                Смысл     Аналитическая роль
  Retail        Invoice Номинальная (ID) Идентификатор заказа   Ключ (внутри Retail)
  Retail      StockCode Номинальная (ID)           Код товара              Измерение
  Retail    Description       Контентная      Описание товара                      —
  Retail       Quantity       Абсолютная    Количество единиц                   Мера
  Retail          Price        Отношений      Цена за единицу                   Мера
  Retail        Revenue        Отношений  Выручка = Qty*Price                   Мера
  Retail    InvoiceDate     Интервальная   Дата/время покупки                  Время
  Retail    Customer ID Номинальная (ID)           ID клиента              Измерение
  Retail        Country   Категориальная               Страна              Измерение
  Retail           Year     Интервальная                  Год           Ключ слияния
  Retail          Month   Категориальная                Месяц    

## 6. Обоснование структуры хранения

**Выбор: аналитическая витрина в Parquet + SQLite (схема «звезда»).**

Обоснование:
- **Parquet** — колоночный формат, эффективен для OLAP-агрегаций и хранит типы (в отличие от CSV). Поддерживает сжатие, читается pandas/pyarrow.
- **SQLite** — для быстрых JOIN и проверки целостности на этапе 2.3–2.5, не требует сервера.
- **Схема «звезда»:** центральная таблица фактов `fact_monthly` (время × источник) + измерения `dim_region`, `dim_store`.
- Слияние — по оси **«Время»** (`Year`, `Month`), как обосновано в 2.2 (OLAP-оси).

**Общие атрибуты для слияния:**
| Источник | Ключ времени | Уровень |
|---|---|---|
| Retail | YearMonth | месяц |
| XML | Year, period | полугодие |
| Walmart | YearMonth | месяц |
| Reviews | review_date | день |

In [21]:
# Retail: месячная выручка
def agg_retail(df):
    g = df.groupby('YearMonth').agg(
        retail_revenue=('Revenue','sum'),
        retail_orders=('Invoice','nunique'),
        retail_customers=('Customer ID','nunique'),
        retail_qty=('Quantity','sum'),
    ).reset_index()
    g['retail_avg_check'] = g['retail_revenue'] / g['retail_orders']
    return g

# Walmart: месячные продажи + доля выбросов
def agg_walmart(df):
    g = df.groupby('YearMonth').agg(
        walmart_sales=('Weekly_Sales','sum'),
        walmart_stores=('Store','nunique'),
        walmart_holiday_share=('Holiday_Flag','mean'),
        walmart_outlier_share=('is_outlier','mean'),
        walmart_avg_temp=('Temperature','mean'),
        walmart_avg_fuel=('Fuel_Price','mean'),
        walmart_avg_cpi=('CPI','mean'),
        walmart_avg_unemp=('Unemployment','mean')
    ).reset_index()
    return g

# XML: годовой оборот — ТОЛЬКО H2 (итог года, без двойного счёта H1+H2)
def agg_xml(df):
    df_h2 = df[df['period'] == 'H2']
    g = df_h2.groupby('year').agg(
        xml_turnover_rub=('value_rub','sum'),
        xml_regions=('region','nunique')
    ).reset_index().rename(columns={'year':'Year'})
    return g

df_retail_month, timings['agg_retail'] = timed('agg_retail', agg_retail, df_retail_clean)
df_walmart_month, timings['agg_walmart'] = timed('agg_walmart', agg_walmart, df_walmart_clean)
df_xml_year, timings['agg_xml'] = timed('agg_xml', agg_xml, df_xml_clean)

print("Retail по месяцам:\n", df_retail_month.head())
print("\nWalmart по месяцам:\n", df_walmart_month.head())
print("\nXML по годам (только H2):\n", df_xml_year)

[TIME] agg_retail: 0.12 сек
[TIME] agg_walmart: 0.01 сек
[TIME] agg_xml: 0.01 сек
Retail по месяцам:
   YearMonth  retail_revenue  retail_orders  retail_customers  retail_qty  retail_avg_check
0   2010-12      570422.730           1400               885      311048        407.444807
1   2011-01      568101.310            987               741      348473        575.583901
2   2011-02      446084.920            997               758      265027        447.427202
3   2011-03      594081.760           1321               974      347582        449.721241
4   2011-04      468374.331           1149               856      291366        407.636493

Walmart по месяцам:
   YearMonth  walmart_sales  walmart_stores  walmart_holiday_share  walmart_outlier_share  walmart_avg_temp  walmart_avg_fuel  walmart_avg_cpi  walmart_avg_unemp
0   2010-02   1.903330e+08              45                   0.25                    0.0         36.288000          2.691967       167.834459           8.619311
1   2010

## 7. Слияние по оси «Время»

Логика:
1. Retail и Walmart — на уровне месяца (`YearMonth`). Внутреннее соединение по общим месяцам (2010–2011).
2. XML — на уровне года (`Year`). Левое соединение к месячной таблице (годовой оборот дублируется на месяцы года как контекст).
3. Reviews в витрину фактов не сливаем — это отдельный контур для NLP (2.4), но сохраняем датасет.

**Важно:** Retail — Великобритания, Walmart — США, XML — РФ. Географически не пересекаются, поэтому слияние чисто по времени — это контекстное обогащение (макро-факторы), а не одна и та же популяция. Это ограничение зафиксировано в 2.1 (границы исследования).

In [22]:
def merge_sources(retail_m, walmart_m, xml_y):
    m = pd.merge(retail_m, walmart_m, on='YearMonth', how='inner')
    m['Year'] = m['YearMonth'].str[:4].astype(int)
    m = pd.merge(m, xml_y, on='Year', how='left')
    m = m.sort_values('YearMonth').reset_index(drop=True)
    return m

df_fact_monthly, timings['merge'] = timed('merge', merge_sources,
                                          df_retail_month, df_walmart_month, df_xml_year)

# Проверка целостности (п.5 методички — скрипт слияния должен валидировать)
assert df_fact_monthly['YearMonth'].is_unique, "Дубли по YearMonth!"
assert df_fact_monthly['xml_turnover_rub'].notna().all(), "Пропуски XML после left join!"
assert len(df_fact_monthly) == 13, f"Ожидалось 13 месяцев, получено {len(df_fact_monthly)}"
print("Проверка целостности пройдена: 13 уникальных месяцев, XML без пропусков")

print("Итоговая витрина fact_monthly:", df_fact_monthly.shape)
print(df_fact_monthly.to_string(index=False))

[TIME] merge: 0.01 сек
Проверка целостности пройдена: 13 уникальных месяцев, XML без пропусков
Итоговая витрина fact_monthly: (13, 17)
YearMonth  retail_revenue  retail_orders  retail_customers  retail_qty  retail_avg_check  walmart_sales  walmart_stores  walmart_holiday_share  walmart_outlier_share  walmart_avg_temp  walmart_avg_fuel  walmart_avg_cpi  walmart_avg_unemp  Year  xml_turnover_rub  xml_regions
  2010-12      570422.730           1400               885      311048        407.444807   288760532.72              45                   0.20               0.066667         39.813556          2.988098       168.459270           8.475289  2010      7.838506e+11           83
  2011-01      568101.310            987               741      348473        575.583901   163703966.83              45                   0.00               0.000000         34.387222          3.113956       168.791818           8.291200  2011      8.216347e+11           83
  2011-02      446084.920            997

In [23]:
t0 = time.time()
df_fact_monthly.to_parquet('fact_monthly.parquet', index=False)
df_reviews_full.to_parquet('reviews_full.parquet', index=False)
df_reviews_clean.to_parquet('reviews_sample_svd.parquet', index=False)
timings['save_parquet'] = time.time() - t0

t0 = time.time()
conn = sqlite3.connect('tobd.db')
df_fact_monthly.to_sql('fact_monthly', conn, if_exists='replace', index=False)
df_xml_clean.to_sql('dim_region_year', conn, if_exists='replace', index=False)
df_walmart_clean.to_sql('dim_store_week', conn, if_exists='replace', index=False)
df_reviews_full[['Id','ProductId','UserId','Score','sentiment','review_date','text_length']] \
    .to_sql('dim_reviews', conn, if_exists='replace', index=False)
conn.close()
timings['save_sqlite'] = time.time() - t0

# Проверка
check = pd.read_parquet('fact_monthly.parquet')
print("Parquet fact_monthly OK:", check.shape)
print("Parquet reviews_full OK:", pd.read_parquet('reviews_full.parquet').shape)
conn = sqlite3.connect('tobd.db')
print("SQLite таблицы:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
conn.close()

Parquet fact_monthly OK: (13, 17)
Parquet reviews_full OK: (568401, 14)
SQLite таблицы:               name
0     fact_monthly
1  dim_region_year
2   dim_store_week
3      dim_reviews


In [24]:
print("Замеры времени (п.5 методички)")
total = 0
for k, v in timings.items():
    print(f"{k:25s}: {v:8.2f} сек")
    total += v
print(f"{'Итого':25s}: {total:8.2f} сек")

Замеры времени (п.5 методички)
load_retail              :    85.20 сек
load_xml                 :     0.18 сек
load_walmart             :     0.09 сек
load_reviews             :     5.80 сек
clean_retail             :     1.59 сек
clean_xml                :     0.01 сек
clean_walmart            :     0.01 сек
tfidf_reviews            :     3.59 сек
svd_reviews              :     0.73 сек
agg_retail               :     0.12 сек
agg_walmart              :     0.01 сек
agg_xml                  :     0.01 сек
merge                    :     0.01 сек
save_parquet             :     4.18 сек
save_sqlite              :     5.11 сек
Итого                    :   106.63 сек


# Выводы по п.2.3

1. **Очистка.** Retail очищен от отмен, отрицательных значений, дубликатов и пропусков `Customer ID` (541 910 → 392 693 строк); добавлена выручка и временные признаки. XML очищен от агрегатов (РФ, федеральные округа), значения переведены в рубли; для годового итога используется **только H2** (январь-декабрь), чтобы избежать двойного счёта H1+H2. Walmart проверен на выбросы (флаг `is_outlier`, 34 выброса). Reviews очищены, обогащены тональностью (`sentiment`), длинами текста; для векторизации взята репрезентативная подвыборка 50 000, TF-IDF (5000) + SVD (20).

2. **Шкалы и роли.** Для всех атрибутов 4 источников определены аналитические шкалы (по 8-шкальной типологии) и роли (измерение/мера/ключ/время/фактор). `Score` трактуется как порядковое измерение, не как мера. Общий атрибут слияния — **время** (`Year`, `Month`/`YearMonth`).

3. **Структура хранения.** Выбрана аналитическая витрина Parquet + SQLite по схеме «звезда»: `fact_monthly` + измерения `dim_region_year`, `dim_store_week`, `dim_reviews`. Артефакты: `fact_monthly.parquet`, `reviews_full.parquet`, `reviews_sample_svd.parquet`, `tobd.db`.

4. **Слияние.** Retail + Walmart соединены внутренним JOIN по `YearMonth` (13 общих месяцев: 2010-12 … 2011-12), XML добавлен левым JOIN по `Year` как годовой макро-контекст. Выполнена валидация целостности (уникальность месяцев, отсутствие пропусков XML). Географическая несовместимость (Retail — UK, Walmart — US, XML — РФ) зафиксирована как ограничение: слияние по времени — контекстное обогащение, а не одна популяция.

5. **Замеры времени** выполнены на всех этапах (загрузка, очистка, агрегация, слияние, сохранение). Основное узкое место — загруз85 XLSX (~91 сек, openpyxl); в 2.5 будет предложена альтернатива (Parquet/CSV) для ускорения.

6. **Подготовка к 2.4.** Для корреляционного анализа между Retail и Walmart меры разного масштаба и валюты — в 2.4 будут стандартизированы (z-score / логарифм) перед совместным анализом.